In [ ]:
from pathlib import Path
import os
import numpy as np
import pandas as pd

ROOT = Path(os.environ.get("PINPAD_DATA_ROOT", "/home/cse/pinpad/myenv/GNURadio_1/Correlation_Responses/OneSubcarrierLowDataRate"))
OUTPUT = Path(os.environ.get("PINPAD_OUTPUT_ROOT", "/home/cse/pinpad/myenv/GNURadio_1/PINPAD_Results"))
OUTPUT.mkdir(parents=True, exist_ok=True)
N_BUTTONS, N_PER_BUTTON = 12, 200
SOURCE = [(c, "R1") for c in range(1, 6)]
ADAPT_CC = [(c, "R1") for c in range(6, 11)]
ADAPT_CR = [(c, "R2") for c in range(1, 6)]
ADAPT_CCR = [(c, "R2") for c in range(11, 16)]
TEST_CC = [(c, "R1") for c in range(16, 26)]
TEST_CR = [(c, "R3") for c in range(1, 6)]
TEST_CCR = [(c, "R2") for c in range(26, 36)]
ADAPT = {"Source": SOURCE, "CC": ADAPT_CC, "CR": ADAPT_CR, "CCR": ADAPT_CCR}
TEST = {"CC": TEST_CC, "CR": TEST_CR, "CCR": TEST_CCR}
assert sum(map(len, ADAPT.values())) == 20
assert sum(map(len, TEST.values())) == 25
assert not set(sum(ADAPT.values(), [])) & set(sum(TEST.values(), []))
assert 20 * 12 * 200 == 48000 and 25 * 12 * 200 == 60000

COLLECTION_MAP = ROOT / "collection_paths.csv"

def _paths():
    if COLLECTION_MAP.exists():
        frame = pd.read_csv(COLLECTION_MAP)
        if not {"card", "reader", "directory"} <= set(frame.columns):
            raise ValueError("collection_paths.csv requires card,reader,directory")
        mapping = {}
        for row in frame.itertuples(index=False):
            key = (int(row.card), str(row.reader).upper())
            if key in mapping:
                raise ValueError(f"Duplicate collection: {key}")
            p = Path(str(row.directory))
            mapping[key] = p if p.is_absolute() else ROOT / p
        return mapping
    mapping = {}
    for reader in ("R1", "R2", "R3"):
        for card in range(1, 36):
            candidates = [ROOT / reader / f"card{card}", ROOT / reader.lower() / f"card{card}",
                          ROOT / f"card{card}" / reader, ROOT / f"card{card}" / reader.lower()]
            matches = [x for x in candidates if x.is_dir()]
            if len(matches) == 1:
                mapping[(card, reader)] = matches[0]
            elif len(matches) > 1:
                raise ValueError(f"Ambiguous collection paths for {card}, {reader}: {matches}")
    return mapping

COLLECTIONS = _paths()
EXPECTED = set(sum(ADAPT.values(), []) + sum(TEST.values(), []))
missing = sorted(EXPECTED - set(COLLECTIONS))
if missing:
    template = OUTPUT / "collection_paths_REQUIRED.csv"
    pd.DataFrame([{"card": c, "reader": r, "directory": ""} for c, r in sorted(EXPECTED)]).to_csv(template, index=False)
    raise FileNotFoundError(f"Missing {len(missing)} card-reader collections: {missing}. Set PINPAD_DATA_ROOT or fill {COLLECTION_MAP} using {template}; reader identity cannot be guessed from card folder names.")

def _read_collection(key):
    p = COLLECTIONS[key]
    parts = []
    for b in range(N_BUTTONS):
        paths = [p / f"button{b}.npy", p / f"Button{b}.npy"]
        hits = [x for x in paths if x.is_file()]
        if len(hits) != 1:
            raise FileNotFoundError(f"Expected one button{b}.npy in {p}; found {hits}")
        x = np.load(hits[0], allow_pickle=False)
        if len(x) != N_PER_BUTTON:
            raise ValueError(f"{hits[0]} has {len(x)} responses; expected {N_PER_BUTTON}")
        if not np.issubdtype(x.dtype, np.number):
            raise TypeError(f"Non-numeric response array: {hits[0]}")
        if x.ndim != 2 or x.shape[0] != N_PER_BUTTON:
            raise ValueError(f"Expected 200 waveform rows: {hits[0]} {x.shape}")
        if not np.isfinite(x).all():
            raise ValueError(f"Non-finite waveform data: {hits[0]}")
        parts.append(x)
    if len({x.shape[1:] for x in parts}) != 1:
        raise ValueError(f"Response shape mismatch: {p}")
    return np.concatenate(parts), np.repeat(np.arange(N_BUTTONS), N_PER_BUTTON)

def load_group(keys):
    data = [_read_collection(k) for k in keys]
    X = np.concatenate([t[0] for t in data]); y = np.concatenate([t[1] for t in data])
    card = np.concatenate([np.full(len(t[1]), k[0]) for k,t in zip(keys,data)])
    reader = np.concatenate([np.full(len(t[1]), k[1]) for k,t in zip(keys,data)])
    return X, y, card, reader

for label, group in {**{f"adapt_{x}":v for x,v in ADAPT.items()}, **{f"test_{x}":v for x,v in TEST.items()}}.items():
    assert len(group) * 2400 in [12000, 24000]
print("Validated collection map:", len(EXPECTED), "collections;", 108000, "expected total responses")

RESPONSE_LENGTH = 3624
SEED = 42


In [ ]:
from sklearn.model_selection import GroupShuffleSplit

def final_split(y, cards, sample_ids):
    groups = np.asarray([f"{c}:{b}:{s // 50}" for c, b, s in zip(cards, y, sample_ids)])
    first, test = next(GroupShuffleSplit(n_splits=1, test_size=0.8, random_state=SEED).split(y, y, groups=groups))
    a, b = next(GroupShuffleSplit(n_splits=1, test_size=0.5, random_state=SEED).split(y[first], y[first], groups=groups[first]))
    cal, val = first[a], first[b]
    assert not (set(groups[cal]) & set(groups[val]) or set(groups[test]) & set(groups[cal]) or set(groups[test]) & set(groups[val]))
    assert 0.08 <= len(cal)/len(y) <= 0.12 and 0.08 <= len(val)/len(y) <= 0.12
    return cal, val, test

def final_data():
    data = {name: load_group(keys) for name, keys in TEST.items()}
    for domain, (X, y, cards, readers) in data.items():
        if X.shape[1] != RESPONSE_LENGTH:
            raise ValueError(f"Unexpected response length in {domain}: {X.shape[1]}")
    return data


In [ ]:
DL_INPUT_FORMAT = os.environ.get("PINPAD_DL_INPUT_FORMAT", "real_imag").lower()
RF_INPUT_FORMAT = os.environ.get("PINPAD_RF_INPUT_FORMAT", "real_imag").lower()
if DL_INPUT_FORMAT not in ("real_imag", "magnitude", "real") or RF_INPUT_FORMAT not in ("real_imag", "magnitude", "real"):
    raise ValueError("Unsupported response representation")

def dl_features(X):
    if DL_INPUT_FORMAT == "real_imag":
        return np.stack([X.real, X.imag], -1).astype(np.float32)
    if DL_INPUT_FORMAT == "magnitude":
        return np.abs(X).astype(np.float32)[..., None]
    return X.real.astype(np.float32)[..., None]

def rf_features(X):
    if RF_INPUT_FORMAT == "real_imag":
        return np.concatenate([X.real, X.imag], axis=1).astype(np.float32)
    if RF_INPUT_FORMAT == "magnitude":
        return np.abs(X).astype(np.float32)
    return X.real.astype(np.float32)


In [ ]:
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
import joblib

source_x, source_y, _, _ = load_group(SOURCE)
target_raw = {name: load_group(keys)[0] for name, keys in (("CC",ADAPT_CC),("CR",ADAPT_CR),("CCR",ADAPT_CCR))}
final_raw = final_data()
source_features = rf_features(source_x)
target_features = {name: rf_features(x) for name,x in target_raw.items()}

COMPONENTS = int(os.environ.get("PINPAD_RF_PCA_COMPONENTS", "64"))
EPS = float(os.environ.get("PINPAD_RF_CORAL_EPS", "0.00001"))
if not 1 <= COMPONENTS <= min(source_features.shape[0],source_features.shape[1]):
    raise ValueError("Invalid PCA dimension")

pca = PCA(n_components=COMPONENTS, random_state=SEED)
source_z = pca.fit_transform(source_features)
target_z = {name: pca.transform(x) for name,x in target_features.items()}

def psd_power(matrix, p):
    eigenvalues, eigenvectors = np.linalg.eigh(matrix)
    return (eigenvectors * np.maximum(eigenvalues, 1e-12)**p) @ eigenvectors.T

mu_source = source_z.mean(axis=0)
mu_target = np.mean([v.mean(axis=0) for v in target_z.values()],axis=0)
c_source = np.cov(source_z,rowvar=False)
c_target = np.mean([np.cov(v,rowvar=False) for v in target_z.values()],axis=0)
identity = np.eye(COMPONENTS)
transform = psd_power(c_source + EPS*identity,-0.5) @ psd_power(c_target + EPS*identity,0.5)
source_aligned = (source_z-mu_source) @ transform + mu_target

rf_source = RandomForestClassifier(n_estimators=300,random_state=SEED,n_jobs=-1).fit(source_z,source_y)
rf = RandomForestClassifier(n_estimators=300,random_state=SEED,n_jobs=-1).fit(source_aligned,source_y)
joblib.dump(dict(pca=pca,rf=rf,rf_source=rf_source,mu_s=mu_source,mu_t=mu_target,a=transform,input_format=RF_INPUT_FORMAT),OUTPUT/"RF_models.joblib")


In [ ]:
rows=[]
for domain,(X,y,cards,readers) in final_raw.items():
    ids = np.tile(np.arange(N_PER_BUTTON), len(TEST[domain])*N_BUTTONS)
    cal,val,test = final_split(y,cards,ids)
    z = pca.transform(rf_features(X[test]))
    for name,model in (("Source",rf_source),("CORAL",rf)):
        probabilities = model.predict_proba(z)
        rows.append(dict(model=name,domain=domain,heldout=len(test),accuracy=float(accuracy_score(y[test],probabilities.argmax(1)))))
        np.savez_compressed(OUTPUT/f"RF_{name}_{domain}_probabilities.npz",probabilities=probabilities,labels=y[test],card=cards[test],reader=readers[test],sample_id=ids[test],heldout_index=test)
summary = pd.DataFrame(rows)
summary.to_csv(OUTPUT/"RF_domain_results.csv",index=False)
print(summary.to_string(index=False))
